# SciPy

NumPy provides arrays and basic linear algebra. SciPy builds on that foundation with algorithms that appear throughout scientific computing: matrix exponentials, ordinary differential equation solvers, function minimizers, and more. For quantum computing, three submodules are essential: `scipy.linalg` for the matrix exponential that generates unitary time evolution, `scipy.integrate` for solving the Schrödinger equation, and `scipy.optimize` for variational quantum algorithms.

This notebook accompanies **Lesson 9** of *Python Programming for Quantum Technology I*. Topics covered:
- The SciPy ecosystem and its relationship to NumPy
- `scipy.linalg`: matrix exponential, eigendecomposition, and matrix functions
- `scipy.integrate`: solving the time-dependent Schrödinger equation
- `scipy.optimize`: variational ground-state search (VQE-style)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Common gate matrices used throughout
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H_gate = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

---
## 1. Linear Algebra: `scipy.linalg`

`scipy.linalg` extends NumPy's linear algebra with functions that are indispensable in quantum computing. The most important is the matrix exponential.

### The Matrix Exponential

The time evolution operator for a time-independent Hamiltonian $H$ is:

$$U(t) = e^{-iHt}$$

`scipy.linalg.expm` computes the matrix exponential of any square matrix. This is **not** the same as applying `np.exp` element-wise: `expm` computes the true matrix function via a Padé approximation.

In [ ]:
# U = exp(-i Z t) for t = pi/2 gives the S gate (up to global phase)
t = np.pi / 2
U = linalg.expm(-1j * Z * t)

print("U(t=π/2) under H=Z:")
print(np.round(U, 6))
print()

# Verify unitarity: U† U = I
print("Unitary?", np.allclose(U.conj().T @ U, np.eye(2)))

In [ ]:
# Evolve |+> under H = Z, plot P(|0>) as a function of time
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)

times = np.linspace(0, 2 * np.pi, 300)
prob_zero = np.array([np.abs((linalg.expm(-1j * Z * t) @ ket_plus)[0])**2 for t in times])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(times, prob_zero, color="steelblue", linewidth=2.5)
ax.axhline(0.5, color="gray", linewidth=1, linestyle=":", alpha=0.7)
ax.set_xlabel(r"Time $t$", fontsize=12)
ax.set_ylabel(r"$P(|0\rangle)$", fontsize=12)
ax.set_title(r"$|{+}\rangle$ evolving under $H = Z$", fontsize=13)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

### Eigendecomposition: Finding the Ground State

`scipy.linalg.eigh` computes eigenvalues and eigenvectors of a Hermitian matrix. It returns eigenvalues in ascending order, so the first column of `eigenvectors` is the ground state.

In [ ]:
# Transverse-field Ising model on two qubits: H = -J*ZZ - h*(XI + IX)
J = 1.0
h = 0.5

ZZ = np.kron(Z, Z)
XI = np.kron(X, I)
IX = np.kron(I, X)

H_ising = -J * ZZ - h * (XI + IX)

eigenvalues, eigenvectors = linalg.eigh(H_ising)

print("Energy levels:", np.round(eigenvalues, 4))
print("Ground state energy:", round(eigenvalues[0], 6))
print("Ground state (dominant components):")
ground_state = eigenvectors[:, 0]
labels = ["|00⟩", "|01⟩", "|10⟩", "|11⟩"]
for label, amp in zip(labels, ground_state):
    if abs(amp) > 0.01:
        print(f"  {label}: {amp:.4f}  (prob = {abs(amp)**2:.4f})")

### Other Matrix Functions

In [ ]:
# sqrtm: the square root of X is the sqrt-NOT gate
sqrtX = linalg.sqrtm(X)
print("sqrtX @ sqrtX == X:", np.allclose(sqrtX @ sqrtX, X))
print("sqrtX is unitary:",   np.allclose(sqrtX.conj().T @ sqrtX, np.eye(2)))
print()

# logm: recover the effective Hamiltonian from a known unitary
# U = exp(-i H t), so H_eff = i * logm(U) / t
t_known = 1.0
U_known = linalg.expm(-1j * Z * t_known)
H_recovered = 1j * linalg.logm(U_known) / t_known

print("Recovered Hamiltonian from U = exp(-iZt):")
print(np.round(H_recovered, 6))
print("Matches Z?", np.allclose(H_recovered, Z))

### Exercise 1.1

**Building a rotation gate from `expm`.**

A rotation around the $Y$ axis by angle $\theta$ is defined as:

$$R_Y(\theta) = e^{-i \theta Y / 2}$$

1. Use `linalg.expm` to compute $R_Y(\theta)$ for $\theta \in \{0, \pi/2, \pi, 3\pi/2, 2\pi\}$.
2. Apply each gate to $|0\rangle$ and compute $P(|1\rangle)$.
3. Plot $P(|1\rangle)$ as a function of $\theta$ from 0 to $2\pi$.
4. Verify that $R_Y(\pi)|0\rangle = |1\rangle$ (up to numerical tolerance).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
ket_0 = np.array([1, 0], dtype=complex)

# Your code here


---
## 2. Solving the Schrödinger Equation: `scipy.integrate`

The time-dependent Schrödinger equation ($\hbar = 1$) is:

$$i\frac{d|\psi\rangle}{dt} = H|\psi\rangle \quad \Rightarrow \quad \frac{d|\psi\rangle}{dt} = -iH|\psi\rangle$$

This is a system of coupled ODEs, one per amplitude. `solve_ivp` solves it from an initial state $|\psi(0)\rangle$.

In [ ]:
from scipy.integrate import solve_ivp

# Derivative function for the Schrödinger equation
def schrodinger(t, psi, H):
    """Time derivative of psi under Hamiltonian H."""
    return -1j * (H @ psi)

# Rabi oscillations: H = (Omega/2) * X, qubit starts in |0>
Omega = 2 * np.pi
H_rabi = (Omega / 2) * X

psi0   = np.array([1, 0], dtype=complex)
t_span = (0, 3)
t_eval = np.linspace(0, 3, 600)

sol = solve_ivp(
    schrodinger,
    t_span,
    psi0,
    t_eval=t_eval,
    args=(H_rabi,),
    method="RK45",
    rtol=1e-9,
    atol=1e-10,
)

# sol.y has shape (n_components, n_timepoints)
print("sol.y shape:", sol.y.shape)
print("Solver message:", sol.message)

In [ ]:
prob_one_ode      = np.abs(sol.y[1])**2
prob_one_analytic = np.sin(Omega * t_eval / 2)**2

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(t_eval, prob_one_ode,      color="steelblue", linewidth=2.5, label="solve_ivp")
ax.plot(t_eval, prob_one_analytic, color="tomato",    linewidth=1.5,
        linestyle="--", label="Analytic (Rabi formula)")

ax.set_xlabel(r"Time (units of $1/\Omega$)", fontsize=12)
ax.set_ylabel(r"$P(|1\rangle)$", fontsize=12)
ax.set_title("Rabi Oscillations: ODE Solver vs Analytic Formula", fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# Verify agreement
max_error = np.max(np.abs(prob_one_ode - prob_one_analytic))
print(f"Maximum error vs analytic: {max_error:.2e}")

### Time-Dependent Hamiltonians

`solve_ivp` handles time-dependent Hamiltonians transparently: make `H` a function of `t` inside the derivative function.

In [ ]:
# Qubit under a constant Z bias + a Gaussian control pulse on X
omega_z      = 1.0
pulse_amp    = 2.0
pulse_center = 1.5
pulse_width  = 0.4

def H_td(t):
    pulse = pulse_amp * np.exp(-0.5 * ((t - pulse_center) / pulse_width)**2)
    return omega_z * Z + pulse * X

def schrodinger_td(t, psi):
    return -1j * (H_td(t) @ psi)

psi0   = np.array([1, 0], dtype=complex)
t_span = (0, 3)
t_eval = np.linspace(0, 3, 600)

sol_td = solve_ivp(schrodinger_td, t_span, psi0, t_eval=t_eval,
                   method="RK45", rtol=1e-9, atol=1e-10)

prob_zero_td = np.abs(sol_td.y[0])**2
prob_one_td  = np.abs(sol_td.y[1])**2

# Pulse envelope for reference
pulse_envelope = np.array([pulse_amp * np.exp(-0.5 * ((t - pulse_center) / pulse_width)**2)
                            for t in t_eval])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax1.plot(t_eval, prob_zero_td, color="steelblue", linewidth=2, label=r"$P(|0\rangle)$")
ax1.plot(t_eval, prob_one_td,  color="tomato",    linewidth=2, label=r"$P(|1\rangle)$", linestyle="--")
ax1.set_ylabel("Probability", fontsize=12)
ax1.set_title("Qubit State Under Gaussian Control Pulse", fontsize=13)
ax1.legend(fontsize=10)
ax1.set_ylim(-0.05, 1.05)

ax2.fill_between(t_eval, pulse_envelope, alpha=0.4, color="goldenrod", label="Pulse envelope")
ax2.set_xlabel("Time", fontsize=12)
ax2.set_ylabel("Pulse amplitude", fontsize=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Exercise 2.1

**Decoherence via an effective non-Hermitian Hamiltonian.**

A simple phenomenological model of amplitude damping adds an imaginary decay term to the Hamiltonian:

$$H_{\text{eff}} = \frac{\Omega}{2} X - i \frac{\Gamma}{2} |1\rangle\langle 1|$$

where $\Gamma$ is the decay rate and $|1\rangle\langle 1| = \begin{pmatrix}0 & 0 \\ 0 & 1\end{pmatrix}$.

1. Build $H_{\text{eff}}$ as a NumPy array for $\Omega = 2\pi$ and $\Gamma = 0.5$.
2. Solve the Schrödinger equation starting from $|0\rangle$ over the interval $t \in [0, 4]$.
3. Normalize the state at each time step (the non-Hermitian Hamiltonian causes norm decay).
4. Plot $P(|0\rangle)$ and $P(|1\rangle)$ on the same axes, both with and without the decay term, for comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

X = np.array([[0, 1], [1, 0]], dtype=complex)

Omega = 2 * np.pi
Gamma = 0.5

# Your code here


---
## 3. Optimization: `scipy.optimize`

Variational quantum algorithms find the minimum of an expected energy:

$$E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle \geq E_0$$

The variational principle guarantees the left side is always an upper bound on the true ground state energy $E_0$. `scipy.optimize.minimize` performs this search.

In [ ]:
from scipy.optimize import minimize

# Single-qubit VQE: H = Z, ansatz |ψ(θ)> = cos(θ/2)|0> + sin(θ/2)|1>
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def ansatz(theta):
    return np.array([np.cos(theta / 2), np.sin(theta / 2)], dtype=complex)

def energy(params):
    psi = ansatz(params[0])
    return np.real(psi.conj() @ Z @ psi)

result = minimize(energy, x0=[0.1], method="BFGS")

print(f"Optimal theta:      {result.x[0]:.6f} rad  (expected: π = {np.pi:.6f})")
print(f"Variational energy: {result.fun:.6f}       (exact ground state: -1.0)")
print(f"Converged:          {result.success}")

### Visualizing the Energy Landscape

Before optimizing, plotting the energy as a function of parameters reveals the landscape: where the minima are, how many there are, and whether the surface is smooth.

In [ ]:
# H = Z + 0.5*X: ground state is not at theta = pi
H_mixed = Z + 0.5 * X

thetas = np.linspace(0, 2 * np.pi, 400)

energies = np.array([
    np.real(ansatz(t).conj() @ H_mixed @ ansatz(t))
    for t in thetas
])

exact_vals = linalg.eigh(H_mixed)[0]
E_min, E_max = exact_vals[0], exact_vals[-1]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(thetas, energies, color="steelblue", linewidth=2.5, label=r"$E(\theta)$")
ax.axhline(E_min, color="tomato",   linewidth=1.5, linestyle="--",
           label=f"Exact min = {E_min:.3f}")
ax.axhline(E_max, color="seagreen", linewidth=1.5, linestyle="--",
           label=f"Exact max = {E_max:.3f}")

ax.set_xlabel(r"$\theta$ (radians)", fontsize=12)
ax.set_ylabel(r"$\langle\psi(\theta)|H|\psi(\theta)\rangle$", fontsize=12)
ax.set_title("Energy Landscape for $H = Z + 0.5X$", fontsize=13)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Two-Parameter Ansatz

A single angle $\theta$ can only reach states on one meridian of the Bloch sphere. The two-parameter ansatz:

$$|\psi(\theta, \phi)\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$$

covers the full Bloch sphere and can represent any single-qubit state.

In [ ]:
# H = Z + 0.5*X: requires a complex ground state, so phi matters
H_mixed = Z + 0.5 * X

def ansatz2(params):
    theta, phi = params
    return np.array([
        np.cos(theta / 2),
        np.exp(1j * phi) * np.sin(theta / 2)
    ], dtype=complex)

def energy2(params):
    psi = ansatz2(params)
    return np.real(psi.conj() @ H_mixed @ psi)

# Try several starting points to be robust against local structure
best = None
for theta_init, phi_init in [(0.5, 0.0), (2.0, 1.0), (1.0, 3.0), (2.5, -1.0)]:
    res = minimize(energy2, [theta_init, phi_init], method="BFGS",
                   options={"gtol": 1e-10})
    if best is None or res.fun < best.fun:
        best = res

exact_E0 = linalg.eigh(H_mixed)[0][0]

print(f"Variational energy:  {best.fun:.10f}")
print(f"Exact ground energy: {exact_E0:.10f}")
print(f"Error:               {abs(best.fun - exact_E0):.2e}")
print(f"Optimal params:      theta={best.x[0]:.4f}, phi={best.x[1]:.4f}")

### Exercise 3.1

**VQE for the two-qubit Ising model.**

The transverse-field Ising Hamiltonian on two qubits is:

$$H = -J\, Z \otimes Z - h\, (X \otimes I + I \otimes X)$$

with $J = 1.0$ and $h = 0.8$.

Use the parameterized ansatz:

$$|\psi(\theta_1, \theta_2)\rangle = R_Y(\theta_1)|0\rangle \otimes R_Y(\theta_2)|0\rangle$$

where $R_Y(\theta) = e^{-i\theta Y/2}$ (compute this with `linalg.expm`).

1. Build the Hamiltonian matrix.
2. Define the ansatz as `np.kron(Ry(theta1) @ ket_0, Ry(theta2) @ ket_0)`.
3. Minimize the expected energy over $(\theta_1, \theta_2)$.
4. Compare the result to the exact ground state energy from `linalg.eigh`.
5. Discuss why this product-state ansatz might not reach the exact ground state for all values of $h$.

In [ ]:
import numpy as np
from scipy import linalg
from scipy.optimize import minimize

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
ket_0 = np.array([1, 0], dtype=complex)

J = 1.0
h = 0.8

# Your code here


---
## Summary

| Submodule | Function | What it computes | Quantum use |
|-----------|----------|-----------------|-------------|
| `scipy.linalg` | `expm(A)` | Matrix exponential $e^A$ | Time evolution $U(t) = e^{-iHt}$ |
| `scipy.linalg` | `eigh(H)` | Eigenvalues/vectors (Hermitian) | Energy levels and eigenstates |
| `scipy.linalg` | `sqrtm(A)` | Matrix square root $\sqrt{A}$ | Gate synthesis (sqrt-NOT, sqrt-SWAP) |
| `scipy.integrate` | `solve_ivp(f, t_span, y0)` | ODE initial-value problem | Schrödinger equation |
| `scipy.optimize` | `minimize(f, x0)` | Minimum of scalar function | Variational quantum eigensolver |

```python
# Common patterns
from scipy import linalg
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

# Time evolution operator
U = linalg.expm(-1j * H * t)

# Ground state energy and state
eigenvalues, eigenvectors = linalg.eigh(H)
E0      = eigenvalues[0]
ground  = eigenvectors[:, 0]

# Schrödinger equation
def deriv(t, psi): return -1j * H @ psi
sol = solve_ivp(deriv, (t0, tf), psi0, t_eval=times)
probs = np.abs(sol.y)**2     # shape: (n_states, n_times)

# Variational minimization
def energy(params):
    psi = build_state(params)
    return np.real(psi.conj() @ H @ psi)
result = minimize(energy, x0, method="BFGS")
```

**What comes next:** This lesson completes *Python Programming for Quantum Technology I*. You now have the full scientific Python toolkit: Python basics, control flow, functions, data structures, modules, OOP, NumPy, Matplotlib, and SciPy. The next course, *Python Programming for Quantum Technology II*, applies these tools to quantum circuits, state simulation, noise modeling, and variational algorithms that target real problems in quantum chemistry and optimization.

---
## Challenge Problem

**A complete quantum annealing simulation.**

Quantum annealing slowly interpolates a Hamiltonian from an easy initial state to a hard problem Hamiltonian, using the adiabatic theorem to track the ground state throughout.

The annealing Hamiltonian is:

$$H(s) = (1 - s)\, H_{\text{driver}} + s\, H_{\text{problem}}$$

where $s = t / T$ goes from 0 to 1 over total time $T$, $H_{\text{driver}} = -(X \otimes I + I \otimes X)$, and $H_{\text{problem}} = Z \otimes Z$.

1. Build both Hamiltonians as 4x4 NumPy arrays.
2. Find the initial ground state of $H(s=0)$ using `linalg.eigh`.
3. Solve the Schrödinger equation for the time-dependent Hamiltonian $H(s(t))$ using `solve_ivp`, for total times $T \in \{1, 5, 20\}$.
4. At the end of each anneal, compute the overlap $|\langle\psi_{\text{final}}|\psi_0^{\text{problem}}\rangle|^2$ with the true ground state of $H_{\text{problem}}$.
5. Plot the ground-state overlap as a function of total anneal time $T$ (try a range from 0.5 to 30). What does the adiabatic theorem predict about the relationship between $T$ and the overlap?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.integrate import solve_ivp

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

H_driver  = -(np.kron(X, I) + np.kron(I, X))
H_problem = np.kron(Z, Z)

def H_anneal(s):
    """Interpolated Hamiltonian at schedule parameter s in [0, 1]."""
    return (1 - s) * H_driver + s * H_problem

# Your code here
